# Image Classification

In [1]:
!pip install -U datasets transformers torchvision accelerate


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import ViTImageProcessor, ViTForImageClassification, TrainingArguments, Trainer
import torchvision.transforms as T

In [ ]:
# dataset
data_dir = "./image-dataset/bottle_cls"
ds = load_dataset("imagefolder", data_dir=data_dir)
labels = ds["train"].features["label"].names   # ['broken_large','broken_small','contamination','good']
id2label = {i: c for i, c in enumerate(labels)}
label2id = {c: i for i, c in enumerate(labels)}

# model + processor
checkpoint = "google/vit-base-patch16-224"
processor = ViTImageProcessor.from_pretrained(checkpoint)

# torchvision transforms (add normalization so ViT sees ImageNet-like inputs)
normalize = T.Normalize(mean=processor.image_mean, std=processor.image_std)
tfm_train = T.Compose([T.Resize((224,224)), T.RandomHorizontalFlip(), T.ToTensor(), normalize])
tfm_eval  = T.Compose([T.Resize((224,224)), T.ToTensor(), normalize])

# dataset transforms
def make_transform(tfm):
    def _f(examples):
        images = [tfm(img.convert("RGB")) for img in examples["image"]]
        return {"pixel_values": images, "label": examples["label"]}
    return _f

ds_train = ds["train"].with_transform(make_transform(tfm_train))
ds_test  = ds["test"].with_transform(make_transform(tfm_eval))

# model
model = ViTForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),               # 4
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,         # replace ImageNet-1k head
)

# training args
args = TrainingArguments(
    output_dir="vit-bottle-4class",
    eval_strategy="epoch",          # corrected arg name
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    learning_rate=3e-5,
    weight_decay=0.05,
    logging_steps=50,
    load_best_model_at_end=True,
)

# collator
def collate(examples):
    pixel_values = torch.stack([e["pixel_values"] for e in examples])
    labels = torch.tensor([e["label"] for e in examples])
    return {"pixel_values": pixel_values, "labels": labels}

# trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    data_collator=collate,
    tokenizer=processor,   # still fine to use this for saving/loading
)

trainer.train()

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [9]:
print(ds)
print(ds["train"].features)

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 209
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 83
    })
})
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['broken_large', 'broken_small', 'contamination', 'good'])}
